# COS Method for Option Pricing — Condensed Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ee2625/fourier-cosine-option-pricing/blob/main/notebooks/demo.ipynb)

This notebook is the trimmed, runs-anywhere version of [`notebooks/tests.ipynb`](https://github.com/ee2625/fourier-cosine-option-pricing/blob/main/notebooks/tests.ipynb). Four sections, each reproducing one published table from Fang & Oosterlee (2008):

- **§1.** Table 1 — density recovery from the characteristic function (standard normal).
- **§2.** Table 2 — Black-Scholes call priced three ways (COS / Lewis / Carr-Madan).
- **§3.** Table 4 — Heston ATM call against the paper reference 5.785155435.
- **§4.** Tables 7 & 8 — Lévy models (Variance Gamma, CGMY).

All paper-replication tables, Bermudan/American convergence, dimensionless-invariance work, and control-variate experiments are in `tests.ipynb`.

### What the COS method is doing

Given a model's characteristic function $\varphi(u) = \mathbb{E}[e^{i u X_T}]$ and a truncation interval $[a,b]$, the COS price of a European option is

$$V(x,0) = K\,e^{-rT}\operatorname{Re}\!\left[\sum_{k=0}^{N-1}{}^{'}\varphi\!\left(\frac{k\pi}{b-a}\right) e^{i k\pi (x-a)/(b-a)}\,V_k\right],$$

where the prime halves the $k=0$ term and the payoff coefficients $V_k$ come from analytic integrals of $\cos\bigl(k\pi\frac{x-a}{b-a}\bigr)$ against the payoff. For smooth densities, cosine coefficients decay exponentially, so error drops geometrically in $N$ until it hits the price-scale double-precision floor (~$2\times 10^{-14}$ when prices are $O(S_0)\approx 100$). Errors displayed as `< 2e-14` below mean *at or under that floor*, not literal zero.

## Install

In [ ]:
!pip install git+https://github.com/ee2625/fourier-cosine-option-pricing.git --quiet

## Imports and shared helpers

`bench` returns (max abs error, mean ms/call). `display_error_table` and `plot_values` floor sub-machine-epsilon errors so log plots and printed tables stay readable.

In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display

from cos_pricing import (
    bsm_price,
    BsmModel, HestonCOSPricer, VgModel, CgmyModel,
    carr_madan_price, lewis_price,
)

pd.set_option("display.float_format", lambda x: f"{x:.6g}")
plt.rcParams["figure.dpi"] = 100

FLOAT_EPS   = np.finfo(float).eps      # 2.22e-16 for IEEE double precision
PRICE_SCALE = 100.0                    # prices are O(S0), with S0 ≈ 100
ERR_FLOOR   = PRICE_SCALE * FLOAT_EPS  # ≈ 2.22e-14; a display floor, not a paper result

def _is_error_col(col):
    label = " ".join(map(str, col)) if isinstance(col, tuple) else str(col)
    return "err" in label.lower()

def display_error_table(df, floor=ERR_FLOOR):
    """Render errors below the price-scale double-precision floor as '< floor'."""
    shown = df.copy()
    for col in shown.columns:
        if _is_error_col(col):
            shown[col] = shown[col].map(
                lambda x: f"< {floor:.0e}" if pd.notna(x) and abs(float(x)) < floor else x
            )
    display(shown)

def plot_values(vals, col, floor=ERR_FLOOR):
    arr = np.asarray(vals, dtype=float)
    return np.maximum(arr, floor) if _is_error_col(col) else arr

def bench(call_fn, ref, reps=50):
    p   = call_fn()
    err = float(np.max(np.abs(np.asarray(p) - np.asarray(ref))))
    t0  = time.perf_counter()
    for _ in range(reps):
        call_fn()
    return err, (time.perf_counter() - t0) / reps * 1e3

print("ready")

## 1. F&O 2008 Table 1 — recovering a density from its CF

Standard normal density on $[-10, 10]$ reconstructed from $N$ cosine coefficients of the characteristic function $\varphi(u) = e^{-u^2/2}$. Errors are evaluated at $x = \pm 5$. Convergence is geometric in $N$ and reaches machine precision by $N = 64$ — this is the underlying mechanism the whole COS pricing method rests on.

In [ ]:
def density_recover(N, a=-10.0, b=10.0, x_eval=np.array([-5.0, 5.0])):
    ba = b - a
    k  = np.arange(N)
    u  = k * np.pi / ba
    Fk = (2.0 / ba) * (np.exp(-0.5 * u**2) * np.exp(-1j * u * a)).real
    Fk[0] *= 0.5
    return np.cos(np.outer((x_eval - a) / ba * np.pi, k)) @ Fk

ref_t1 = norm.pdf(np.array([-5.0, 5.0]))
rows = [(N, float(np.max(np.abs(density_recover(N) - ref_t1))))
        for N in [4, 8, 16, 32, 64]]
df_t1 = pd.DataFrame(rows, columns=["N", "max |err|"])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogy(df_t1["N"], plot_values(df_t1["max |err|"], "max |err|"),
            "o-", color="#1f77b4")
ax.set(xlabel="N", ylabel="max |err|",
       title="Table 1 — density recovery")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout(); plt.show()
display_error_table(df_t1)

## 2. F&O 2008 Table 2 — BSM call, three Fourier pricers

Black-Scholes call with $\sigma = 0.25$, $r = 0.1$, $T = 0.1$, $S = 100$, strikes $K \in \{80, 100, 120\}$. We price three ways and compare against the analytic BSM formula:

- **COS** — density cosine expansion on $[a,b]$ with analytic payoff integrals.
- **Lewis** — single Fourier integral on the contour $u - i/2$, no damping parameter.
- **Carr-Madan** — FFT of a damped call-price transform; needs the damping $\alpha$ and converges algebraically here.

COS hits the price-scale floor by $N \approx 64$. Lewis converges quickly but a few orders behind. Carr-Madan is dramatically worse on this short-dated setup — exactly the regime F&O 2008 designed Table 2 to expose.

In [ ]:
sig, r, q, T, S = 0.25, 0.1, 0.0, 0.1, 100.0
strikes_t2 = np.array([80.0, 100.0, 120.0])
ref_t2 = bsm_price(strikes_t2, S, sig, T, intr=r, divr=q, cp=+1)

bsm  = BsmModel(sigma=sig, intr=r, divr=q)
fwd2 = S * np.exp((r - q) * T)
df2  = np.exp(-r * T)
cf_bsm = lambda u: np.exp(-0.5 * sig**2 * T * u * (u + 1j))   # CF of log(S_T/F)

rows = []
for N in [32, 64, 128, 256, 512]:
    cos_e, cos_ms = bench(lambda: bsm.price(strikes_t2, S, T, cp=+1, n_cos=N), ref_t2)
    lw_e,  lw_ms  = bench(lambda: lewis_price(cf_bsm, T, strikes_t2, fwd2, df2,
                                              cp=+1, n_quad=N), ref_t2)
    cm_e,  cm_ms  = bench(lambda: carr_madan_price(cf_bsm, T, strikes_t2, fwd2, df2,
                                                   cp=+1, N=N, eta_grid=100.0/N), ref_t2)
    rows.append((N, cos_e, cos_ms, lw_e, lw_ms, cm_e, cm_ms))

df_t2 = pd.DataFrame(rows, columns=["N",
    "COS err", "COS ms", "Lewis err", "Lewis ms", "CM err", "CM ms"])

methods  = [("COS", "o-", "#1f77b4"),
            ("Lewis", "^-", "#2ca02c"),
            ("Carr-Madan", "s-", "#ff7f0e")]
err_cols = ["COS err", "Lewis err", "CM err"]
ms_cols  = ["COS ms", "Lewis ms", "CM ms"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for (lbl, st, col), ec, mc in zip(methods, err_cols, ms_cols):
    a1.semilogy(df_t2["N"], plot_values(df_t2[ec], ec), st, label=lbl, color=col)
    a2.plot    (df_t2["N"], df_t2[mc], st, label=lbl, color=col)
a1.set(xlabel="N", ylabel="max |err|", title="Error vs N")
a1.grid(True, which="both", alpha=0.3); a1.legend()
a2.set(xlabel="N", ylabel="ms / call", title="Runtime vs N")
a2.grid(True, alpha=0.3); a2.legend()
fig.suptitle("Table 2 — BSM: three Fourier pricers", y=1.02)
fig.tight_layout(); plt.show()
display_error_table(df_t2)

## 3. F&O 2008 Table 4 — Heston, ATM, T = 1

Heston stochastic-volatility model with the published Table 4 parameters: $v_0 = 0.0175$, $\lambda = 1.5768$, $\eta = 0.5751$, $\bar v = 0.0398$, $\rho = -0.5711$, $r = q = 0$. The headline paper reference for the $K = 100$, $T = 1$ call is

$$V^{\text{paper}} = 5.785155435.$$

Heston has no closed-form density, only a characteristic function — exactly the case COS is designed for. Convergence here is still geometric but with a milder rate than BSM, since the Heston log-price density is less analytically pleasant.

In [ ]:
PAPER = dict(S0=100.0, v0=0.0175, lam=1.5768, eta=0.5751,
             ubar=0.0398, rho=-0.5711, r=0.0, q=0.0)
K_h, T_h = 100.0, 1.0
REF_h    = 5.785155435    # F&O 2008 Table 4, T=1

heston = HestonCOSPricer(**PAPER)
rows = []
for N in [40, 80, 120, 160, 200, 240]:
    _, ms = bench(lambda: heston.price_call(K_h, T_h, N=N), REF_h)
    price = float(heston.price_call(K_h, T_h, N=N))
    rows.append((N, price, abs(price - REF_h), ms))
df_t4 = pd.DataFrame(rows, columns=["N", "COS price", "|err|", "ms"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.semilogy(df_t4["N"], plot_values(df_t4["|err|"], "|err|"),
            "o-", color="#1f77b4")
a1.axhline(ERR_FLOOR, color="#888", linestyle="--", linewidth=1.0,
           label=f"price-scale floor (~{ERR_FLOOR:.0e})")
a1.set(xlabel="N", ylabel="|err vs paper ref|",
       title=f"Heston, T={T_h}, K={K_h:g}")
a1.grid(True, which="both", alpha=0.3); a1.legend()

a2.plot(df_t4["N"], df_t4["ms"], "o-", color="#1f77b4")
a2.set(xlabel="N", ylabel="ms / call", title="Runtime vs N")
a2.grid(True, alpha=0.3)
fig.suptitle(f"Table 4 — Heston ATM call, paper ref = {REF_h}", y=1.02)
fig.tight_layout(); plt.show()
display_error_table(df_t4)

## 4. F&O 2008 Tables 7 & 8 — Lévy models (VG and CGMY)

Two pure-jump Lévy models. Variance Gamma runs Brownian motion on a random Gamma clock; CGMY is a four-parameter generalization where $(C, G, M, Y)$ control jump activity, left-tail decay, right-tail decay, and small-jump fine structure.

- **VG:** $\sigma = 0.12$, $\theta = -0.14$, $\nu = 0.2$, $r = 0.1$, $K = 90$, $T = 1$. Reference is a self-consistent COS price at $N = 2^{14}$.
- **CGMY:** $C = 1$, $G = M = 5$, $Y = 0.5$, $r = 0.1$, $K = S = 100$, $T = 1$. Paper reference (Table 8) is $19.8129487706$.

In [ ]:
S0_l, T_l, R_l = 100.0, 1.0, 0.1

vg   = VgModel(sigma=0.12, theta=-0.14, nu=0.2, intr=R_l, divr=0.0)
cgmy = CgmyModel(C=1.0, G=5.0, M=5.0, Y=0.5, intr=R_l, divr=0.0)

K_vg, K_cg = 90.0, 100.0
ref_vg     = float(vg.price(K_vg, S0_l, T_l, cp=+1, n_cos=2**14))   # self-consistent
REF_cgmy   = 19.8129487706                                          # F&O Table 8

rows_vg = [(N, abs(float(vg.price(K_vg, S0_l, T_l, cp=+1, n_cos=N)) - ref_vg))
           for N in [30, 60, 90, 120, 150]]
rows_cg = [(N, abs(float(cgmy.price(K_cg, S0_l, T_l, cp=+1, n_cos=N)) - REF_cgmy))
           for N in [40, 60, 80, 100, 120, 140]]

df_vg = pd.DataFrame(rows_vg, columns=["N", "VG |err|"])
df_cg = pd.DataFrame(rows_cg, columns=["N", "CGMY |err|"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.semilogy(df_vg["N"], plot_values(df_vg["VG |err|"], "|err|"),
            "o-", color="#1f77b4")
a1.axhline(ERR_FLOOR, color="#888", linestyle="--", linewidth=1.0)
a1.set(xlabel="N", ylabel="|err|",
       title=f"Table 7 — VG, K={K_vg:g}, ref={ref_vg:.6f}")
a1.grid(True, which="both", alpha=0.3)

a2.semilogy(df_cg["N"], plot_values(df_cg["CGMY |err|"], "|err|"),
            "s-", color="#2ca02c")
a2.axhline(ERR_FLOOR, color="#888", linestyle="--", linewidth=1.0)
a2.set(xlabel="N", ylabel="|err|",
       title=f"Table 8 — CGMY Y=0.5, paper ref={REF_cgmy:g}")
a2.grid(True, which="both", alpha=0.3)

fig.suptitle("Pure-jump Lévy models — paper-grid convergence", y=1.02)
fig.tight_layout(); plt.show()

print("Variance Gamma (Table 7):")
display_error_table(df_vg)
print("CGMY (Table 8):")
display_error_table(df_cg)

## What's in the full notebook

`notebooks/tests.ipynb` covers the rest of the paper-replication suite and our project's own extensions:

- F&O 2008 Table 3 — cash-or-nothing digital, Tables 5–6 — Heston at $T=10$ and 21-strike sweep, Tables 9–10 — CGMY at $Y=1.5$ and $Y=1.98$.
- F&O 2009 — Bermudan put converging to its European-limit and American-limit.
- Buckingham-π / dimensionless-invariance work — 3D surfaces showing that BSM and Heston prices collapse onto a single dimensionless surface under spot/time rescalings.
- Strike-independent reusable setup, Junike-Pankrashkin truncation ranges, and Black-Scholes control-variate experiments.

### References
- Fang, F. & Oosterlee, C. W. (2008). *A novel pricing method for European options based on Fourier-cosine series expansions*. SIAM J. Sci. Comput., 31(2), 826–848.
- Fang, F. & Oosterlee, C. W. (2009). *Pricing early-exercise and discrete barrier options by Fourier-cosine series expansions*. Numer. Math., 114(1), 27–62.